---

## 프로세스 요약 표

| STEP 01. Data Prep | STEP 02. Model Design | STEP 03. Setup | STEP 04. Training | STEP 05. Evaluation |
| :---: | :---: | :---: | :---: | :---: |
| **데이터 수집 & 전처리** | **`nn.Module` 구조 정의** | **손실함수 & 옵티마이저** | **5단계 학습 루프** | **성능 검증 & 평가** |
| `StandardScaler`<br>`DataLoader` | `Linear(30→32→16→1)`<br>`ReLU` / `Sigmoid` | `BCELoss`<br>`Adam (lr=0.001)` | `Forward` ➔ `Loss`<br>➔ `Zero` ➔ `Back` ➔ `Step` | `model.eval()`<br>`torch.no_grad()` |

---

## 📝 단계별 세부 프로세스 내용

### STEP 01. 데이터 로드 및 전처리
* **데이터셋 로드 및 분할**: Scikit-Learn 유방암 데이터셋(30개 특성)을 Train(80%)과 Test(20%)로 분할
* **특성 스케일링**: `StandardScaler`를 적용하여 데이터의 평균을 0, 표준편차를 1로 표준화
* **텐서(Tensor) 변환**: NumPy 배열을 PyTorch `float32` 텐서로 변환 (타겟 데이터 차원: `(N, 1)`)
* **미니배치 구성**: `TensorDataset`과 `DataLoader`를 사용해 `batch_size=32` 단위로 묶고 셔플(Shuffle) 설정

### STEP 02. 신경망 모델 클래스 정의
* **`nn.Module` 상속**: 커스텀 클래스 `BinaryClassifier` 정의
* **`__init__()` (레이어 초기화)**:
  * **Layer 1**: `nn.Linear(30, 32)` + `nn.ReLU()`
  * **Layer 2**: `nn.Linear(32, 16)` + `nn.ReLU()`
  * **Output Layer**: `nn.Linear(16, 1)` + `nn.Sigmoid()`
* **`forward()` (순전파 흐름)**: 입력 데이터 `x`가 은닉층을 거쳐 0~1 사이의 확률값으로 출력되는 연산 정의

### STEP 03. 손실 함수 및 옵티마이저 설정
* **손실 함수 (Loss Function)**: `nn.BCELoss()` (Binary Cross Entropy) — 이진 분류 예측값과 정답 간의 오차 측정
* **옵티마이저 (Optimizer)**: `optim.Adam(model.parameters(), lr=0.001)` — 효율적인 경사하강법 기반 가중치 최적화

### STEP 04. 모델 학습 루프 실행 (Training)
* **모드 전환**: `model.train()` 호출로 학습 모드 설정
* **미니배치 단위 5단계 학습 순서**:
  1. **Forward Pass**: `outputs = model(batch_X)` (예측값 계산)
  2. **Loss Calculation**: `loss = criterion(outputs, batch_y)` (손실값 측정)
  3. **Zero Grad**: `optimizer.zero_grad()` (이전 기울기 리셋)
  4. **Backward Pass**: `loss.backward()` (역전파로 기울기 계산)
  5. **Optimizer Step**: `optimizer.step()` (가중치 갱신)

### STEP 05. 모델 평가 및 성능 측정 (Evaluation)
* **평가 모드 전환**: `model.eval()`을 호출하여 DropOut/BatchNorm 동작 고정
* **기울기 계산 비활성화**: `with torch.no_grad():` 블록을 사용하여 메모리 절약 및 연산 속도 향상
* **임계값 적용**: Output 확률값이 `0.5` 이상이면 Class 1, 미만이면 Class 0으로 이진 분류
* **최종 평가**: 테스트 데이터셋의 **Test Loss** 및 **Accuracy(정확도 %)** 산출

In [1]:
import sklearn

# [Windows에서 아래 코드를 실행하면 오류발생]:
# UnicodeDecodeError: 'cp949' codec can't decodeXXX
# model = LogisticRegression(max_iter=10000 )

# [해결방법]
# HTML 다이어그램 출력 기능을 끄고 텍스트로 표시
import platform
print(platform.system())
if platform.system()== 'Windows':
   sklearn.set_config(display='text')

import platform
import matplotlib.pyplot as plt

# OS에 따른 한글 폰트 설정
if platform.system() == 'Windows':
    plt.rc('font', family='Malgun Gothic')  # 윈도우: 맑은 고딕
elif platform.system() == 'Darwin':
    plt.rc('font', family='AppleGothic')    # 맥: 애플 고딕
else:
    plt.rc('font', family='NanumBarunGothic') # 리눅스

# 마이너스 기호 깨짐 방지
plt.rc('axes', unicode_minus=False)

import pandas as pd
import warnings



warnings.filterwarnings(action='ignore')  # 경고 메시지를 무시하고 숨기기  warnings.filterwarnings('ignore')
# warnings.filterwarnings(action='default')   # 숨긴 경고 메시지 다시 보이기

#모든 컬럼들이 보이도록 설정한다.
pd.options.display.max_columns=829
# pd.options.display.max_columns = None  # 제한 없이 모든 열 표시

Windows


In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [6]:
# ==========================================
# 1. 데이터셋 로드 및 전처리
# ==========================================

# 사이킷런에서 유방암(Breast Cancer) 이진 분류 데이터셋 로드
data = load_breast_cancer()
X, y = data.data, data.target # X: 특성(Feature) 데이터, y: 정답 라벨(Target, 0 또는 1)

# 학습용 데이터(80%)와 테스트용 데이터(20%) 분할
# random_state를 지정하여 실행 시마다 동일하게 분할되도록 설정
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 특성 스케일링 (StandardScaler: 평균 0, 표준편차 1로 표준화)
# 신경망 학습의 안정성과 속도를 위해 필수적인 단계
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train) # 학습 데이터 기준으로 스케일러 학습(fit) 및 변환(transform)
X_test = scaler.transform(X_test)       # 테스트 데이터는 학습된 스케일러 기준으로 변환만 수행

# 넘파이(NumPy) 배열을 파이토치 텐서(Tensor)로 변환
# PyTorch 연산은 float32 형태의 텐서를 기본으로 사용함
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
print(f'X_train_tensor:{X_train_tensor.shape}')
# y의 차원을 (N,)에서 (N, 1)로 변경 (.unsqueeze(1)) -> 모델 출력 차원(N, 1)과 일치시킴
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
print(f'y_train_tensor:{y_train_tensor.shape}')
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1)

# TensorDataset: X(입력)와 y(타겟)를 하나의 dataset 객체로 묶어 관리
dataset = TensorDataset(X_train_tensor, y_train_tensor)

# DataLoader: 미니배치(Mini-batch) 단위로 데이터를 묶고 셔플(Shuffle)하는 역할
# batch_size=32: 한 번에 32개씩 데이터를 묶어서 모델에 입력
# shuffle=True: 매 에포크(Epoch)마다 데이터 순서를 섞어 과적합 방지
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)


X_train_tensor:torch.Size([455, 30])
y_train_tensor:torch.Size([455, 1])


In [7]:
# ==========================================
# 2. nn.Module을 상속받는 신경망 모델 클래스 정의
# ==========================================

class BinaryClassifier(nn.Module):
    def __init__(self, input_dim):
        """
        초기화 메서드: 모델에 사용할 레이어(층) 및 연산 요소를 선언합니다.
        input_dim: 입력 데이터의 특성(Feature) 개수 (유방암 데이터는 30개)
        """
        super(BinaryClassifier, self).__init__() # 부모 클래스(nn.Module) 초기화
        
        # 선형 레이어(Fully Connected Layer) 정의
        self.fc1 = nn.Linear(input_dim, 32) # 입력 차원(input_dim) -> 은닉층 1 차원(32)
        self.fc2 = nn.Linear(32, 16)        # 은닉층 1 차원(32) -> 은닉층 2 차원(16)
        self.fc3 = nn.Linear(16, 1)         # 은닉층 2 차원(16) -> 출력 차원(1: 이진 분류)
        
        # 활성화 함수(Activation Function) 정의
        self.relu = nn.ReLU()       # 비선형성을 부여하는 ReLU 함수
        self.sigmoid = nn.Sigmoid() # 출력을 0~1 사이의 확률값으로 변환하는 Sigmoid 함수

    def forward(self, x):
        """
        순전파(Forward Pass) 연산: 입력 데이터 x가 층을 통과하는 흐름을 정의합니다.
        """
        x = self.relu(self.fc1(x))    # fc1 -> ReLU
        x = self.relu(self.fc2(x))    # fc2 -> ReLU
        x = self.sigmoid(self.fc3(x)) # fc3 -> Sigmoid (최종 예측 확률)
        return x

# 모델 인스턴스 생성
input_dim = X_train.shape[1] # 특성 개수 (30)
model = BinaryClassifier(input_dim)

print("--- [모델 구조] ---")
print(model)
print("--------------------\n")

--- [모델 구조] ---
BinaryClassifier(
  (fc1): Linear(in_features=30, out_features=32, bias=True)
  (fc2): Linear(in_features=32, out_features=16, bias=True)
  (fc3): Linear(in_features=16, out_features=1, bias=True)
  (relu): ReLU()
  (sigmoid): Sigmoid()
)
--------------------

